<a href="https://colab.research.google.com/github/tamzinzanalcock/Latin-Dictionary/blob/main/Latin_Dictionary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cltk

In [ ]:
import os
os.environ["STANZA_RESOURCES_DIR"] = os.path.expanduser("~/stanza_resources")

import stanza
stanza.download("la")

from cltk.data.fetch import FetchCorpus
FetchCorpus(language="lat").import_corpus("lat_models_cltk")

In [ ]:
from cltk.nlp import NLP
from cltk.lexicon.lat import LatinLewisLexicon
import unicodedata

nlp = NLP(language="lat", suppress_banner=True)
lexicon = LatinLewisLexicon()

def strip_macrons(text):
    """Remove macrons (and other diacritics) so input matches CLTK's expected unmarked forms."""
    normalized = unicodedata.normalize('NFD', text)
    return ''.join(ch for ch in normalized if unicodedata.category(ch) != 'Mn')

def short_def(full_definition):
    """Keep just the headword, principal parts, and core gloss (before the first colon)."""
    if not full_definition:
        return ""
    return full_definition.split(":")[0].strip()

def define_word(word):
    word = strip_macrons(word)
    doc = nlp.analyze(text=word)
    if not doc.words or not doc.words[0].lemma:
        return f"Couldn't find a lemma for '{word}'"
    lemma = doc.words[0].lemma

    definition = short_def(lexicon.lookup(lemma))
    if definition:
        return definition

    # Lemmatizer may have failed silently — try the raw word too
    definition = short_def(lexicon.lookup(word))
    if definition:
        return definition

    return f"No definition found for '{word}' (lemma guessed as '{lemma}')"

In [ ]:
print("Latin dictionary lookup — type a word to look it up, or 'quit' to stop.\n")

while True:
    word = input("Latin word: ").strip()
    if word.lower() in ("quit", "exit", ""):
        print("Goodbye!")
        break
    print(define_word(word))
    print()